<a href="https://colab.research.google.com/github/Sangeetha3315/Agentic-AI-and-computer-vision-workshop-projects/blob/main/AI_research_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q groq ddgs beautifulsoup4 requests

In [ ]:
import os, getpass

# Prompt the user to enter their Groq API key securely.
# The input is then stored as an environment variable named 'GROQ_API_KEY' for the current session.
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")
# Print a confirmation message to the user.
print("Key set for this session.")

In [ ]:
"""
Core agent logic for the Groq + DuckDuckGo research agent.
Kept in a standalone module so it can be unit-tested outside the notebook.
"""
import json
import os
import time
import textwrap

import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
from groq import Groq

MODEL = "openai/gpt-oss-120b"
MAX_STEPS = 6

SYSTEM_PROMPT = textwrap.dedent("""\
    You are a careful research agent. You have two tools: web_search and fetch_page.

    Rules:
    - For any question involving current events, recent data, specific facts, or
      anything that could have changed since you were trained, use web_search first.
      Do not answer from memory alone on those topics.
    - Use fetch_page when a search snippet isn't enough detail and you need the
      full page content.
    - Call tools as many times as needed (multiple searches are fine and encouraged
      for multi-part questions), but be efficient — don't repeat an identical search.
    - When you have enough information, respond with your final answer as plain text
      starting with "FINAL ANSWER:". Include a short "Sources:" list of the URLs you
      actually used at the end.
    - If tools return errors or empty results, try a reworded query before giving up.
""")

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web via DuckDuckGo and return titles, URLs, and snippets.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query."},
                    "max_results": {
                        "type": "integer",
                        "description": "Number of results to return (default 5, max 10).",
                    },
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_page",
            "description": "Fetch a web page and return its visible text content (truncated).",
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {"type": "string", "description": "Absolute URL to fetch."},
                },
                "required": ["url"],
            },
        },
    },
]


def web_search(query: str, max_results: int = 5) -> str:
    max_results = max(1, min(int(max_results or 5), 10))
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
    except Exception as e:
        return json.dumps({"error": f"search failed: {e}"})

    if not results:
        return json.dumps({"error": "no results found"})

    formatted = [
        {"title": r.get("title", ""), "url": r.get("href", ""), "snippet": r.get("body", "")}
        for r in results
    ]
    return json.dumps(formatted, ensure_ascii=False)


def fetch_page(url: str, max_chars: int = 3000) -> str:
    try:
        resp = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0 (research-agent-demo)"},
            timeout=10,
        )
        resp.raise_for_status()
    except Exception as e:
        return json.dumps({"error": f"fetch failed: {e}"})

    soup = BeautifulSoup(resp.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()
    text = " ".join(soup.get_text(separator=" ").split())
    truncated = text[:max_chars]
    return json.dumps({"url": url, "content": truncated}, ensure_ascii=False)


TOOL_FUNCTIONS = {"web_search": web_search, "fetch_page": fetch_page}


def _call_with_retry(client, **kwargs):
    delay = 2
    for attempt in range(5):
        try:
            return client.chat.completions.create(**kwargs)
        except Exception as e:
            msg = str(e).lower()
            if "rate limit" in msg or "429" in msg:
                print(f"  (rate limited, waiting {delay}s...)")
                time.sleep(delay)
                delay *= 2
                continue
            raise
    raise RuntimeError("Exceeded retries due to repeated rate limiting.")


def run_agent(question: str, api_key: str = None, model: str = MODEL,
              max_steps: int = MAX_STEPS, verbose: bool = True) -> str:
    """Run the ReAct-style tool-calling loop and return the final answer text."""
    api_key = api_key or os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise ValueError("No Groq API key found. Set GROQ_API_KEY or pass api_key=.")

    client = Groq(api_key=api_key)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for step in range(1, max_steps + 1):
        if verbose:
            print(f"\n{'=' * 60}\nSTEP {step}\n{'=' * 60}")

        response = _call_with_retry(
            client,
            model=model,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0.2,
        )
        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append(msg)
            for tool_call in msg.tool_calls:
                name = tool_call.function.name
                try:
                    args = json.loads(tool_call.function.arguments or "{}")
                except json.JSONDecodeError:
                    args = {}

                if verbose:
                    print(f"[tool call] {name}({args})")

                fn = TOOL_FUNCTIONS.get(name)
                result = fn(**args) if fn else json.dumps({"error": f"unknown tool {name}"})

                if verbose:
                    preview = result[:300] + ("..." if len(result) > 300 else "")
                    print(f"[tool result] {preview}")

                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": name,
                        "content": result,
                    }
                )
        else:
            final = msg.content or ""
            if verbose:
                print(f"\n[agent finished after {step} step(s)]")
            return final

    return "The agent used all available steps without reaching a final answer. Try increasing max_steps."

In [ ]:
# Call the run_agent function with a specific research question.
# The 'verbose=True' argument ensures that all intermediate steps (tool calls, results) are printed.
answer = run_agent(
    "What are the most significant AI model releases announced in the last month, "
    "and what's notable about each?",
    verbose=True,
)
# Print a separator line for better readability.
print("\n" + "=" * 60)
# Print a label for the final answer.
print("FINAL ANSWER")
# Print another separator line.
print("=" * 60)
# Print the final answer returned by the agent.
print(answer)

In [ ]:
# Prompt the user to type in a research question and store their input in the 'question' variable.
question = input("Ask a research question: ")
# Call the run_agent function, passing the user's 'question' and enabling verbose output.
answer = run_agent(question, verbose=True)
# Print a separator line for better readability.
print("\n" + "=" * 60)
# Print a label for the final answer.
print("FINAL ANSWER")
# Print another separator line.
print("=" * 60)
# Print the final answer obtained from the agent based on the user's question.
print(answer)